In [1]:
# the purpose of this notebook is to confirm that the most uncertain formulas are ones outside of the training data 
# formulation space

In [2]:
# import the data

from pathlib import Path
import pandas as pd

# Load the compiled data
project_root = Path.cwd().resolve().parent if 'notebooks' in str(Path.cwd()) else Path.cwd().resolve()
excel_path = project_root / 'data' / 'raw' / 'Compiled Data.xlsx'

df = pd.read_excel(excel_path)

formula_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]
measured_target_cols = [
    "Avg_Density_g_cm3",
    "24h_avg_pct absorption",
    "24h_avg_pct_retention",
]
measured_se_cols = [
    "Avg_Density SE",
    "24h_se.1",
    "24h_se",
]

# Keep each target beside its standard error column
measured_cols = [
    col
    for target, se in zip(measured_target_cols, measured_se_cols)
    for col in (target, se)
]
df = df.loc[:, formula_cols + measured_cols]
df = df.rename(columns={
    "24h_se.1": "se_absorption",
    "24h_se": "se_retention",
})

print(df.head())

   TEM  Joncryl  NC513  PEG-DE  NGDE  Avg_Density_g_cm3  Avg_Density SE  \
0    8      NaN    NaN     NaN   NaN           0.192000        0.002667   
1    8      0.7    NaN     NaN   NaN           0.195000        0.007263   
2    8      1.0    NaN     NaN   NaN           0.159000        0.005033   
3    8      1.5    NaN     NaN   NaN           0.159933        0.004410   
4    8      2.0    NaN     NaN   NaN           0.201000        0.008817   

   24h_avg_pct absorption  se_absorption  24h_avg_pct_retention  se_retention  
0              113.616943       5.271375                  31.77          2.81  
1              103.817309       3.496766                  33.12          2.57  
2              129.464292       5.490086                  36.25          1.77  
3              120.749405       2.660519                  25.33          0.38  
4              113.871581       2.071998                  26.58          0.99  


In [3]:
# get the 100,000 candidate formulas, they are stored in candidate_set

# Formula constraints:
# - TEM: 3-30%.
# - Each chain extender (Joncryl, NC513, PEG-DE, NGDE): 0.7-9% when included; 0% when omitted.
# - Combined chain extender amount: no more than 20%.
# - Generate 100,000 valid candidate formulas.

import numpy as np
import pandas as pd

candidate_count = 100_000
chain_extender_cols = ["Joncryl", "NC513", "PEG-DE", "NGDE"]
feature_cols = ["TEM", *chain_extender_cols]
rng = np.random.default_rng(42)

candidate_chunks = []
collected = 0
while collected < candidate_count:
    batch_size = max(10_000, 2 * (candidate_count - collected))
    active_counts = rng.integers(1, len(chain_extender_cols) + 1, size=batch_size)
    active_order = rng.random((batch_size, len(chain_extender_cols))).argsort(axis=1)
    active_mask = active_order < active_counts[:, None]
    extender_amounts = rng.uniform(0.7, 9.0, size=(batch_size, len(chain_extender_cols)))
    extender_amounts = np.where(active_mask, extender_amounts, 0.0)

    valid_mask = extender_amounts.sum(axis=1) <= 20.0
    valid_batch = extender_amounts[valid_mask]
    candidate_chunks.append(valid_batch)
    collected += len(valid_batch)

extender_values = np.concatenate(candidate_chunks, axis=0)[:candidate_count]
candidate_set = pd.DataFrame(extender_values, columns=chain_extender_cols)
candidate_set.insert(0, "TEM", rng.uniform(3.0, 30.0, size=candidate_count))
candidate_set = candidate_set[feature_cols]

# Confirm the generated formulas satisfy every requested bound.
assert len(candidate_set) == candidate_count
assert candidate_set["TEM"].between(3.0, 30.0).all()
assert candidate_set[chain_extender_cols].sum(axis=1).le(20.0).all()
for col in chain_extender_cols:
    active = candidate_set[col].ne(0)
    assert candidate_set.loc[active, col].between(0.7, 9.0).all()

print(f"Generated {len(candidate_set):,} valid candidate formulas.")
print(candidate_set.head())
print("Maximum total chain extender:", candidate_set[chain_extender_cols].sum(axis=1).max())

Generated 100,000 valid candidate formulas.
         TEM   Joncryl     NC513    PEG-DE      NGDE
0  25.484578  0.000000  0.000000  6.108175  0.000000
1  18.407481  4.615083  1.014021  2.552998  0.790089
2   7.553238  1.243114  5.282248  8.651054  0.000000
3  10.545875  7.095499  0.000000  2.359188  0.000000
4   9.618866  5.856375  0.000000  0.000000  1.028519
Maximum total chain extender: 19.99986528911906


In [4]:
# get predictions and predictive uncertainty for these formulas, stored in
# candidate_predictions

import joblib
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
models_dir = project_root / "models"
gpr_models = joblib.load(models_dir / "gpr_models.joblib")

feature_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]
X_candidates = candidate_set.loc[:, feature_cols]

batch_size = 5_000
prediction_batches = {name: {"mean": [], "std": []} for name in gpr_models}

for name, model in gpr_models.items():
    for start in range(0, len(X_candidates), batch_size):
        batch = X_candidates.iloc[start : start + batch_size]
        mean_batch, std_batch = model.predict(batch, return_std=True)
        prediction_batches[name]["mean"].append(mean_batch)
        prediction_batches[name]["std"].append(std_batch)

prediction_columns = {}
for name, batches in prediction_batches.items():
    prediction_columns[f"pred_{name}"] = np.concatenate(batches["mean"])
    prediction_columns[f"std_{name}"] = np.concatenate(batches["std"])

predictions = pd.DataFrame(prediction_columns, index=candidate_set.index)
candidate_predictions = pd.concat([candidate_set, predictions], axis=1)

required_std_cols = [f"std_{name}" for name in gpr_models]
missing_std_cols = [col for col in required_std_cols if col not in candidate_predictions.columns]
if missing_std_cols:
    raise KeyError(f"Prediction standard-deviation columns were not created: {missing_std_cols}")

display(candidate_predictions.head())


,TEM,Joncryl,NC513,PEG-DE,NGDE,pred_Avg_Density_g_cm3,std_Avg_Density_g_cm3,pred_24h_avg_pct absorption,std_24h_avg_pct absorption,pred_24h_avg_pct_retention,std_24h_avg_pct_retention
0,25.484578,0.000000,0.000000,6.108175,0.000000,0.214560,0.063039,149.879372,70.873801,29.664809,13.944631
1,18.407481,4.615083,1.014021,2.552998,0.790089,0.214560,0.063039,150.018094,70.864489,31.236214,13.976423
2,7.553238,1.243114,5.282248,8.651054,0.000000,0.208497,0.062699,149.882627,70.873789,31.258685,13.976317
3,10.545875,7.095499,0.000000,2.359188,0.000000,0.214560,0.063039,149.289656,70.855396,31.248348,13.976498
4,9.618866,5.856375,0.000000,0.000000,1.028519,0.214559,0.063039,144.290398,69.345716,31.248225,13.976498


In [5]:
# find the 10 most uncertain formulas (highest std)

import numpy as np

std_cols = [col for col in candidate_predictions.columns if col.startswith("std_")]
if not std_cols:
    raise ValueError("No standard-deviation columns (std_...) found in candidate_predictions.")

std_values = candidate_predictions[std_cols].to_numpy(dtype=float)
if not np.isfinite(std_values).all():
    raise ValueError("Standard-deviation values contain missing or non-finite values.")

std_min = std_values.min(axis=0)
std_range = std_values.max(axis=0) - std_min

# Normalize each property's uncertainty to [0, 1].
normalized_std = np.divide(
    std_values - std_min,
    std_range,
    out=np.zeros_like(std_values),
    where=std_range > 0,
)

df = candidate_predictions.copy()
df["uncertainty_score"] = normalized_std.mean(axis=1)

# The 10 formulas with the highest equal-weight normalized uncertainty.
df = df.nlargest(10, "uncertainty_score").copy()

display(df)

,TEM,Joncryl,NC513,PEG-DE,NGDE,pred_Avg_Density_g_cm3,std_Avg_Density_g_cm3,pred_24h_avg_pct absorption,std_24h_avg_pct absorption,pred_24h_avg_pct_retention,std_24h_avg_pct_retention,uncertainty_score
47444,28.976835,8.727749,0.000000,8.687788,0.000000,0.21456,0.063039,149.843232,70.87381,31.248348,13.976498,1.0
79500,28.875272,8.883727,0.000000,8.558663,1.011866,0.21456,0.063039,149.843232,70.87381,31.248348,13.976498,1.0
47301,29.959049,8.461572,0.000000,8.446472,0.000000,0.21456,0.063039,149.843234,70.87381,31.248348,13.976498,1.0
17012,29.055218,8.942468,0.813891,8.303461,0.968689,0.21456,0.063039,149.843234,70.87381,31.248348,13.976498,1.0
28935,28.415470,8.491789,0.000000,8.600037,0.000000,0.21456,0.063039,149.843239,70.87381,31.248348,13.976498,1.0
22328,29.341019,8.614419,0.000000,8.189749,0.937476,0.21456,0.063039,149.843238,70.87381,31.248348,13.976498,1.0
53390,28.082666,8.849499,0.000000,8.460884,0.000000,0.21456,0.063039,149.843238,70.87381,31.248348,13.976498,1.0
62522,26.208222,8.863533,0.000000,8.972416,0.000000,0.21456,0.063039,149.843239,70.87381,31.248348,13.976498,1.0
41482,28.632481,7.799037,1.670431,8.905200,0.000000,0.21456,0.063039,149.843243,70.87381,31.248348,13.976498,1.0
9899,28.548926,8.466877,0.000000,8.280847,2.809902,0.21456,0.063039,149.843240,70.87381,31.248348,13.976498,1.0
